In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, max, avg, min, desc
from dotenv import load_dotenv
import os

load_dotenv()
POSTGRESQL_JDBC_PATH=os.getenv('POSTGRESQL_JDBC_PATH')
USER_POSTGRESQL=os.getenv('USER_POSTGRESQL')
PASSWORD_POSTGRESQL=os.getenv('PASSWORD_POSTGRESQL')
URL_POSTGRES=os.getenv('URL_POSTGRES')
SPARK_MASTER_IP=os.getenv('SPARK_MASTER_IP')

In [2]:
spark = SparkSession.builder\
        .appName('Spark_DB_Quering')\
        .master(f'spark://{SPARK_MASTER_IP}:7077')\
        .config('spark.jars', POSTGRESQL_JDBC_PATH)\
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/26 15:46:04 WARN Utils: Your hostname, MacBook-Air-Mihail.local, resolves to a loopback address: 127.0.0.1; using 10.132.33.16 instead (on interface en0)
26/08/26 15:46:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/08/26 15:46:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/mihailmoroz/.pyenv/versions/env1/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
def read_and_cache_table(table_name):
    df = spark.read\
          .format("jdbc")\
          .option("url", URL_POSTGRES)\
          .option("dbtable", table_name)\
          .option("user", USER_POSTGRESQL)\
          .option("password", PASSWORD_POSTGRESQL)\
          .option("driver", "org.postgresql.Driver")\
          .load()

    cached_df = df.cache()

    return cached_df

table_names = ["category","film_category","actor","film_actor","inventory","rental","payment","address","city"]
dataframes = {}
for table in table_names:
    dataframes[table] = read_and_cache_table(table_name=table)


In [4]:
df_movie_num_per_category = dataframes["category"].join(dataframes["film_category"], dataframes["category"].category_id == dataframes["film_category"].category_id, "inner")\
                            .groupBy(dataframes["category"].category_id, dataframes["category"].name).agg(count(dataframes["category"].name).alias('cnt'))\
                            .sort(desc('cnt')).select('name', 'cnt')

df_movie_num_per_category.show()

+-----------+---+
|       name|cnt|
+-----------+---+
|     Sports| 74|
|    Foreign| 73|
|     Family| 69|
|Documentary| 68|
|  Animation| 66|
|     Action| 64|
|        New| 63|
|      Drama| 62|
|     Sci-Fi| 61|
|      Games| 61|
|   Children| 60|
|     Comedy| 58|
|   Classics| 57|
|     Travel| 57|
|     Horror| 56|
|      Music| 51|
+-----------+---+



In [8]:
from pyspark.sql.functions import concat, lit

df_top10_actors = dataframes["actor"].join(dataframes["film_actor"], dataframes["actor"].actor_id == dataframes["film_actor"].actor_id, "left")
df_top10_actors = df_top10_actors.join(dataframes["inventory"], dataframes["inventory"].film_id == df_top10_actors.film_id, "left")
df_top10_actors = df_top10_actors.join(dataframes["rental"], dataframes["rental"].inventory_id == df_top10_actors.inventory_id, "left")\
                  .groupBy(dataframes["actor"].actor_id, 'first_name', 'last_name')\
                  .agg(count('rental_id').alias('movie_cnt'))\
                  .sort(desc('movie_cnt'))\
                  .select('actor_id', concat('first_name', lit(' '), 'last_name').alias('name'), 'movie_cnt')

df_top10_actors.limit(10).show()

+--------+------------------+---------+
|actor_id|              name|movie_cnt|
+--------+------------------+---------+
|     107|    Gina Degeneres|      753|
|     181|    Matthew Carrey|      678|
|     198|       Mary Keitel|      674|
|     144|Angela Witherspoon|      654|
|     102|       Walter Torn|      640|
|      60|       Henry Berry|      612|
|     150|       Jayne Nolte|      611|
|      37|        Val Bolger|      605|
|      23|     Sandra Kilmer|      604|
|      90|      Sean Guiness|      599|
+--------+------------------+---------+



In [12]:
from pyspark.sql.functions import coalesce, sum
df_most_spent_category = dataframes["category"].join(dataframes["film_category"], dataframes["category"].category_id == dataframes["film_category"].category_id, "left")
df_most_spent_category = df_most_spent_category.join(dataframes["inventory"], dataframes["inventory"].film_id == df_most_spent_category.film_id, "left")
df_most_spent_category = df_most_spent_category.join(dataframes["rental"], dataframes["rental"].inventory_id == df_most_spent_category.inventory_id, "left")
df_most_spent_category = df_most_spent_category.join(dataframes["payment"], dataframes["payment"].rental_id == df_most_spent_category.rental_id, "left")

df_most_spent_category = df_most_spent_category.groupBy(dataframes["category"].category_id, dataframes["category"].name)\
                         .agg(coalesce(sum(df_most_spent_category.amount), lit(0)).alias('amt_spent'))\
                         .sort(desc('amt_spent'))

df_most_spent_category.limit(1).show()

+-----------+------+---------+
|category_id|  name|amt_spent|
+-----------+------+---------+
|         15|Sports|  4892.19|
+-----------+------+---------+

